# Module 4B: Record Investigation and Remediation

## Learning Objectives
- Build a formal DQ Issues Log with lifecycle tracking
- Auto-scan tables and log all violations with details
- Implement a quarantine pattern (flag bad records, exclude from Gold)
- Create a resolution workflow with audit trail
- Build an issues dashboard for DQ stewards

## Key Concept: From Detection to Action

Modules 1-4 DETECT quality issues. This module teaches what happens NEXT:

```
DMF detects violation --> Issue logged --> Steward investigates --> Resolution
                              |                    |                    |
                         DQ_ISSUES_LOG        Root cause        FIXED / ACCEPTED /
                         (automated)          analysis          QUARANTINED
```

> **Business Value:** Detection without remediation is just noise. A formal issue log with SLAs turns DQ alerts into an accountable workflow -- issues get owners, deadlines, and resolution tracking.

---

> **Role:** `CORP_DQ_ADMIN` | **Time:** ~45 minutes

> **What this does:** Sets your session context to the lab role, database, and warehouse.

In [ ]:
USE ROLE CORP_DQ_ADMIN;
USE DATABASE CORP_DWH;
USE WAREHOUSE COMPUTE_WH;

---
## 4B-a. Create the DQ Issues Log

This table tracks every quality issue through its lifecycle: OPEN -> INVESTIGATING -> RESOLVED.

In [ ]:
CREATE OR REPLACE TABLE CORP_DWH.DQ.DQ_ISSUES_LOG (
    ISSUE_ID NUMBER AUTOINCREMENT,
    -- What failed
    TABLE_NAME STRING NOT NULL,
    COLUMN_NAME STRING,
    RULE_NAME STRING,
    RECORD_IDENTIFIER STRING COMMENT 'Primary key or unique identifier of the failing record',
    FAILURE_REASON STRING,
    METRIC_VALUE NUMBER COMMENT 'The DMF value that triggered this issue',
    SEVERITY STRING DEFAULT 'MEDIUM',
    -- Lifecycle
    STATUS STRING DEFAULT 'OPEN' COMMENT 'OPEN, INVESTIGATING, FIXED, ACCEPTED, QUARANTINED',
    ASSIGNED_TO STRING,
    RESOLUTION_NOTES STRING,
    -- Audit
    DETECTED_AT TIMESTAMP_LTZ DEFAULT CURRENT_TIMESTAMP(),
    UPDATED_AT TIMESTAMP_LTZ DEFAULT CURRENT_TIMESTAMP(),
    RESOLVED_AT TIMESTAMP_LTZ,
    DETECTED_BY STRING DEFAULT CURRENT_USER(),
    -- Linkage
    SOURCE_SYSTEM STRING,
    DQ_DOMAIN STRING COMMENT 'Accuracy, Completeness, Uniqueness, Freshness, Validity, Volume, Consistency'
);

---
## 4B-b. Auto-Scan and Log Issues

This procedure scans Silver customers and logs every violation it finds:

In [ ]:
CREATE OR REPLACE PROCEDURE CORP_DWH.DQ.LOG_DQ_ISSUES()
RETURNS STRING
LANGUAGE SQL
COMMENT = 'Idempotent: only logs NEW issues not already tracked as OPEN/INVESTIGATING'
AS
DECLARE
    issues_logged NUMBER DEFAULT 0;
BEGIN
    -- Log invalid National IDs (skip if already tracked)
    INSERT INTO CORP_DWH.DQ.DQ_ISSUES_LOG
        (TABLE_NAME, COLUMN_NAME, RULE_NAME, RECORD_IDENTIFIER, FAILURE_REASON, SEVERITY, SOURCE_SYSTEM, DQ_DOMAIN)
    SELECT
        'SILVER.INT_CUSTOMERS',
        'NATIONAL_ID',
        'National ID Format',
        src.CUSTOMER_NAME,
        CASE
            WHEN LEN(src.NATIONAL_ID) < 10 THEN 'Too short (' || LEN(src.NATIONAL_ID) || ' digits, need 10)'
            WHEN LEN(src.NATIONAL_ID) > 10 THEN 'Too long (' || LEN(src.NATIONAL_ID) || ' digits, max 10)'
            WHEN RLIKE(src.NATIONAL_ID, '[A-Za-z]') THEN 'Contains letters (must be numeric only)'
            WHEN NOT STARTSWITH(src.NATIONAL_ID, '1') AND NOT STARTSWITH(src.NATIONAL_ID, '2') THEN 'Must start with 1 or 2'
            ELSE 'Invalid format'
        END,
        'CRITICAL',
        src.SOURCE_SYSTEM,
        'Accuracy'
    FROM CORP_DWH.SILVER.INT_CUSTOMERS src
    WHERE src.NATIONAL_ID IS NOT NULL AND NOT RLIKE(src.NATIONAL_ID, '^[12][0-9]{9}$')
      AND NOT EXISTS (
        SELECT 1 FROM CORP_DWH.DQ.DQ_ISSUES_LOG ex
        WHERE ex.RECORD_IDENTIFIER = src.CUSTOMER_NAME
          AND ex.RULE_NAME = 'National ID Format'
          AND ex.STATUS IN ('OPEN', 'INVESTIGATING')
      );

    issues_logged := issues_logged + SQLROWCOUNT;

    -- Log invalid IBANs (skip if already tracked)
    INSERT INTO CORP_DWH.DQ.DQ_ISSUES_LOG
        (TABLE_NAME, COLUMN_NAME, RULE_NAME, RECORD_IDENTIFIER, FAILURE_REASON, SEVERITY, SOURCE_SYSTEM, DQ_DOMAIN)
    SELECT
        'SILVER.INT_CUSTOMERS',
        'IBAN',
        'IBAN Format',
        src.CUSTOMER_NAME,
        CASE
            WHEN NOT STARTSWITH(src.IBAN, 'SA') THEN 'Does not start with SA'
            WHEN LEN(src.IBAN) != 24 THEN 'Wrong length (' || LEN(src.IBAN) || ' chars, need 24)'
            ELSE 'Invalid characters'
        END,
        'HIGH',
        src.SOURCE_SYSTEM,
        'Accuracy'
    FROM CORP_DWH.SILVER.INT_CUSTOMERS src
    WHERE src.IBAN IS NOT NULL AND NOT RLIKE(src.IBAN, '^SA[0-9A-Za-z]{22}$')
      AND NOT EXISTS (
        SELECT 1 FROM CORP_DWH.DQ.DQ_ISSUES_LOG ex
        WHERE ex.RECORD_IDENTIFIER = src.CUSTOMER_NAME
          AND ex.RULE_NAME = 'IBAN Format'
          AND ex.STATUS IN ('OPEN', 'INVESTIGATING')
      );

    issues_logged := issues_logged + SQLROWCOUNT;

    -- Log NULL National IDs (completeness, skip if already tracked)
    INSERT INTO CORP_DWH.DQ.DQ_ISSUES_LOG
        (TABLE_NAME, COLUMN_NAME, RULE_NAME, RECORD_IDENTIFIER, FAILURE_REASON, SEVERITY, SOURCE_SYSTEM, DQ_DOMAIN)
    SELECT
        'SILVER.INT_CUSTOMERS',
        'NATIONAL_ID',
        'National ID Completeness',
        src.CUSTOMER_NAME,
        'Missing National ID (NULL) -- cannot verify identity',
        'HIGH',
        src.SOURCE_SYSTEM,
        'Completeness'
    FROM CORP_DWH.SILVER.INT_CUSTOMERS src
    WHERE src.NATIONAL_ID IS NULL
      AND NOT EXISTS (
        SELECT 1 FROM CORP_DWH.DQ.DQ_ISSUES_LOG ex
        WHERE ex.RECORD_IDENTIFIER = src.CUSTOMER_NAME
          AND ex.RULE_NAME = 'National ID Completeness'
          AND ex.STATUS IN ('OPEN', 'INVESTIGATING')
      );

    issues_logged := issues_logged + SQLROWCOUNT;

    -- Log duplicates (skip if already tracked)
    INSERT INTO CORP_DWH.DQ.DQ_ISSUES_LOG
        (TABLE_NAME, COLUMN_NAME, RULE_NAME, RECORD_IDENTIFIER, FAILURE_REASON, SEVERITY, SOURCE_SYSTEM, DQ_DOMAIN)
    SELECT
        'SILVER.INT_CUSTOMERS',
        'NATIONAL_ID',
        'Duplicate Detection',
        src.CUSTOMER_NAME,
        'Duplicate record: same National ID exists from another source',
        'CRITICAL',
        src.SOURCE_SYSTEM,
        'Uniqueness'
    FROM CORP_DWH.SILVER.INT_CUSTOMERS src
    WHERE src.IS_DUPLICATE = TRUE
      AND NOT EXISTS (
        SELECT 1 FROM CORP_DWH.DQ.DQ_ISSUES_LOG ex
        WHERE ex.RECORD_IDENTIFIER = src.CUSTOMER_NAME
          AND ex.RULE_NAME = 'Duplicate Detection'
          AND ex.STATUS IN ('OPEN', 'INVESTIGATING')
      );

    issues_logged := issues_logged + SQLROWCOUNT;

    RETURN 'Logged ' || :issues_logged || ' NEW DQ issues (existing open issues skipped).';
END;

---
## Run the Issue Scanner

> **What this does:** Executes the LOG_DQ_ISSUES procedure and shows the result.

In [ ]:
CALL CORP_DWH.DQ.LOG_DQ_ISSUES();

---
## View the Issues Log

> **What this does:** Displays all logged issues sorted by severity, showing the full detail of each violation detected by the scanner.

In [ ]:
SELECT ISSUE_ID, TABLE_NAME, COLUMN_NAME, RECORD_IDENTIFIER,
    FAILURE_REASON, SEVERITY, STATUS, DQ_DOMAIN, SOURCE_SYSTEM
FROM CORP_DWH.DQ.DQ_ISSUES_LOG
ORDER BY SEVERITY DESC, DETECTED_AT DESC;

---
## 4B-c. Issues Dashboard: Summary by Domain and Severity

> **What this does:** Summarizes issues by DQ domain and severity to give stewards a high-level view of where problems are concentrated.

In [ ]:
-- Issues by DQ Domain
SELECT DQ_DOMAIN, SEVERITY, COUNT(*) AS ISSUE_COUNT, STATUS
FROM CORP_DWH.DQ.DQ_ISSUES_LOG
GROUP BY DQ_DOMAIN, SEVERITY, STATUS
ORDER BY CASE SEVERITY WHEN 'CRITICAL' THEN 1 WHEN 'HIGH' THEN 2 WHEN 'MEDIUM' THEN 3 ELSE 4 END;

> **What this does:** Breaks down open issues by source system to identify which upstream system is generating the most quality problems.

In [ ]:
-- Issues by source system (where is the problem coming from?)
SELECT SOURCE_SYSTEM, COUNT(*) AS ISSUES, 
    COUNT(CASE WHEN SEVERITY = 'CRITICAL' THEN 1 END) AS CRITICAL,
    COUNT(CASE WHEN SEVERITY = 'HIGH' THEN 1 END) AS HIGH
FROM CORP_DWH.DQ.DQ_ISSUES_LOG
WHERE STATUS = 'OPEN'
GROUP BY SOURCE_SYSTEM
ORDER BY ISSUES DESC;

---
## 4B-d. Quarantine Pattern

Instead of deleting bad records, we **quarantine** them: mark them so Gold layer excludes them, but they remain available for investigation.

The quarantine uses a STATUS field. Gold queries filter: `WHERE STATUS != 'QUARANTINED'`

In [ ]:
-- Quarantine all records with CRITICAL severity issues
UPDATE CORP_DWH.DQ.DQ_ISSUES_LOG
SET STATUS = 'QUARANTINED',
    UPDATED_AT = CURRENT_TIMESTAMP(),
    RESOLUTION_NOTES = 'Auto-quarantined: CRITICAL severity, pending source system fix'
WHERE SEVERITY = 'CRITICAL' AND STATUS = 'OPEN';

> **What this does:** Queries the issues log to display all records that have been quarantined, showing their identifiers and reasons.

In [ ]:
-- View quarantined records
SELECT RECORD_IDENTIFIER, FAILURE_REASON, SOURCE_SYSTEM, RESOLUTION_NOTES
FROM CORP_DWH.DQ.DQ_ISSUES_LOG
WHERE STATUS = 'QUARANTINED';

---
## 4B-e. Resolution Workflow

When a steward investigates and fixes (or accepts) an issue:

In [ ]:
-- Create a resolution procedure
CREATE OR REPLACE PROCEDURE CORP_DWH.DQ.RESOLVE_ISSUE(
    P_ISSUE_ID NUMBER,
    P_STATUS STRING,
    P_NOTES STRING
)
RETURNS STRING
LANGUAGE SQL
AS
BEGIN
    UPDATE CORP_DWH.DQ.DQ_ISSUES_LOG
    SET STATUS = :P_STATUS,
        RESOLUTION_NOTES = :P_NOTES,
        UPDATED_AT = CURRENT_TIMESTAMP(),
        RESOLVED_AT = CASE WHEN :P_STATUS IN ('FIXED', 'ACCEPTED') THEN CURRENT_TIMESTAMP() ELSE NULL END
    WHERE ISSUE_ID = :P_ISSUE_ID;

    RETURN 'Issue ' || :P_ISSUE_ID || ' updated to ' || :P_STATUS;
END;

> **What this does:** Calls the RESOLVE_ISSUE procedure to mark a National ID Completeness issue as ACCEPTED with an explanation.

In [ ]:
-- Example: Accept the CRM NULL National IDs (known limitation, tracked separately)
CALL CORP_DWH.DQ.RESOLVE_ISSUE(
    (SELECT MIN(ISSUE_ID) FROM CORP_DWH.DQ.DQ_ISSUES_LOG WHERE RULE_NAME = 'National ID Completeness' AND STATUS != 'ACCEPTED'),
    'ACCEPTED',
    'Known CRM limitation. National ID not mandatory in Salesforce. Tracked in JIRA-4521.'
);

---
## 4B-f. Aged Issues Report

Find issues that have been open too long (SLA breach):

In [ ]:
SELECT
    ISSUE_ID,
    RECORD_IDENTIFIER,
    FAILURE_REASON,
    SEVERITY,
    DATEDIFF(HOUR, DETECTED_AT, CURRENT_TIMESTAMP()) AS HOURS_OPEN,
    CASE
        WHEN SEVERITY = 'CRITICAL' AND DATEDIFF(HOUR, DETECTED_AT, CURRENT_TIMESTAMP()) > 4 THEN 'SLA BREACH (>4h)'
        WHEN SEVERITY = 'HIGH' AND DATEDIFF(HOUR, DETECTED_AT, CURRENT_TIMESTAMP()) > 24 THEN 'SLA BREACH (>24h)'
        WHEN SEVERITY = 'MEDIUM' AND DATEDIFF(HOUR, DETECTED_AT, CURRENT_TIMESTAMP()) > 72 THEN 'SLA BREACH (>72h)'
        ELSE 'Within SLA'
    END AS SLA_STATUS
FROM CORP_DWH.DQ.DQ_ISSUES_LOG
WHERE STATUS IN ('OPEN', 'INVESTIGATING')
ORDER BY SEVERITY DESC, DETECTED_AT ASC;

---
## Checkpoint: Remediation Workflow

> **What this does:** Verifies your work so far. All checks should show [PASS].

In [ ]:
from snowflake.snowpark.context import get_active_session
session = get_active_session()

print("=" * 60)
print("CHECKPOINT: Record Investigation + Remediation")
print("=" * 60)
passed = 0

# Issues logged
total = session.sql("SELECT COUNT(*) AS C FROM CORP_DWH.DQ.DQ_ISSUES_LOG").collect()[0]['C']
print(f"  [PASS] DQ_ISSUES_LOG has {total} issues logged")
passed += 1

# By status
statuses = session.sql(
    "SELECT STATUS, COUNT(*) AS C FROM CORP_DWH.DQ.DQ_ISSUES_LOG GROUP BY STATUS ORDER BY STATUS"
).to_pandas()
for _, row in statuses.iterrows():
    print(f"    {row['STATUS']}: {int(row['C'])}")

# Procedure exists
try:
    session.sql("DESCRIBE PROCEDURE CORP_DWH.DQ.LOG_DQ_ISSUES()").collect()
    print(f"  [PASS] LOG_DQ_ISSUES() procedure exists")
    passed += 1
except:
    print("  [FAIL] LOG_DQ_ISSUES() not found")

try:
    session.sql("DESCRIBE PROCEDURE CORP_DWH.DQ.RESOLVE_ISSUE(NUMBER, STRING, STRING)").collect()
    print(f"  [PASS] RESOLVE_ISSUE() procedure exists")
    passed += 1
except:
    print("  [FAIL] RESOLVE_ISSUE() not found")

print(f"\nResult: {passed}/3 checks passed")
print("=" * 60)

---
## Quiz: Test Your Knowledge

**Q1:** Why do we QUARANTINE records instead of deleting them?

**Q2:** An issue is marked ACCEPTED with notes "Known CRM limitation." What does this mean for the DQ score?

**Q3:** The aged issues report shows a CRITICAL issue open for 6 hours (SLA is 4h). What should happen?

**Q4:** A new data load arrives and the same National ID violation is detected again. Should the procedure log a DUPLICATE issue, or skip it?

> **What this does:** Reveals quiz answers. Try answering first!

In [ ]:
print("""
QUIZ ANSWERS
============

Q1: We quarantine instead of deleting because:
    - Deleting loses audit trail (you can't prove the issue existed)
    - The record may be partially valid (good email, bad National ID)
    - Source system may need the record for reconciliation
    - Regulatory requirements may mandate data retention
    - You need the record to investigate root cause
    Quarantine = excluded from Gold (business doesn't see it) but preserved for investigation.

Q2: ACCEPTED means the team has reviewed the issue, confirmed it's a known
    limitation (not a bug), and decided not to fix it. The DQ score still reflects
    the violation (it's real!), but no alert fires and no SLA applies.
    Common for: source systems that can't be changed, legacy data, known trade-offs.

Q3: SLA breach on CRITICAL = escalation:
    1. Alert should have fired (Module 7)
    2. Assigned steward gets notified
    3. If still unresolved, escalate to data owner (per RULES_CATALOG.OWNER)
    4. If > 8 hours, auto-quarantine to prevent bad data from reaching Gold
    This is why we combine DQ_ISSUES_LOG + Alerts + Ownership.

Q4: The procedure should be IDEMPOTENT: check if an open issue already exists
    for the same record+rule combination before inserting. If it exists, skip.
    Otherwise you get noise from repeated scans.
    Enhancement: add WHERE NOT EXISTS (SELECT 1 FROM DQ_ISSUES_LOG WHERE 
    RECORD_IDENTIFIER = X AND RULE_NAME = Y AND STATUS IN ('OPEN','INVESTIGATING'))
""")

---
## Summary

| Component | Purpose |
|-----------|---------|
| DQ_ISSUES_LOG | Formal issue tracking with lifecycle |
| LOG_DQ_ISSUES() | Auto-scan and log all violations |
| Quarantine pattern | Exclude bad records from Gold without deleting |
| RESOLVE_ISSUE() | Record resolution with audit trail |
| Aged Issues Report | SLA compliance monitoring |

**Key Insight:** The complete DQ workflow is: Detect (DMFs) -> Log (Issues) -> Investigate (Drill-down) -> Resolve (Fix/Accept/Quarantine) -> Prevent (fix source system). Without this workflow, detection is just noise.

---

**Next:** Open `5_AI_ML_DQ` for AI-powered quality checks.